In [11]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


SETUP: Create expanded sample company data (50+ chunks)

In [3]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")

Created 53 sample chunks for demonstration


In [ ]:
# Convert to Document objects for LangChain
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

SETUP: Create the three types of retrievers

# 1. Vector Retriever (Semantic Search/Dense Retrieval)

In [5]:
print("Setting up hybrid retriever...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

Setting up hybrid retriever...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12663.07it/s]


# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [6]:
# 2. BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 15

# 3. Hybrid Retriever

In [7]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

print("Setup complete!\n")

Setup complete!



Step 1: Get results from hybrid search (retrieve more chunks)

In [8]:
query = "Tesla financial performance and production updates"

print("STEP 1: Hybrid Search Results (Top 10)")
print("-" * 50)

retrieved_docs = hybrid_retriever.invoke(query)  # Get top 25 for reranking

# Show top 10 from hybrid search
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print(f"\n(Retrieved {len(retrieved_docs)} total chunks for reranking)\n")

STEP 1: Hybrid Search Results (Top 10)
--------------------------------------------------
 1. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 2. Tesla announced plans to expand Gigafactory production capacity.
 3. Tesla reported strong free cash flow generation of $7.5 billion.
 4. Tesla stock price reached new highs following earnings announcement.
 5. Tesla continues to lead in electric vehicle market share globally.
 6. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 7. Tesla Model Y became the best-selling vehicle worldwide.
 8. The Tesla coil was invented by Nikola Tesla in 1891.
 9. Tesla's automotive gross margin improved to 19.3% this quarter.
10. Tesla's energy storage business grew 40% year-over-year.
11. Tesla's Full Self-Driving revenue increased significantly.
12. Stock market analysts upgrade technology sector ratings.
13. Financial markets showed volatility during earnings season.
14. Production delays affected multiple man

Step 2: Apply Cross-Encoder Reranking

In [13]:
print("STEP 2: After Cross-Encoder Reranking (Top 10)")
print("-" * 50)

# Initialize the cross-encoder reranker (local, no API key needed)
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=10)

# Rerank the retrieved documents
reranked_docs = reranker.compress_documents(retrieved_docs, query)

# Show reranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print("\n" + "=" * 80)
print("ANALYSIS:")
print("✅ Hybrid Search: Mixed relevant and irrelevant results")
print("✅ Reranking: Most relevant Tesla financial/production info at top")
print("✅ Notice how reranking moved the most contextually relevant chunks higher")

# Optional: Show the difference more clearly
print("\n" + "=" * 80)
print("KEY IMPROVEMENTS AFTER RERANKING:")
print("-" * 40)


hybrid_top_3 = [doc.page_content for doc in retrieved_docs[:5]]
reranked_top_3 = [doc.page_content for doc in reranked_docs[:5]]

print("BEFORE (Hybrid Top 3):")
for i, content in enumerate(hybrid_top_3, 1):
    print(f"  {i}. {content}")

print("\nAFTER (Reranked Top 3):")
for i, content in enumerate(reranked_top_3, 1):
    print(f"  {i}. {content}")

STEP 2: After Cross-Encoder Reranking (Top 10)
--------------------------------------------------


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10748.33it/s]


 1. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 2. Tesla announced plans to expand Gigafactory production capacity.
 3. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 4. Tesla reported strong free cash flow generation of $7.5 billion.
 5. Tesla's automotive gross margin improved to 19.3% this quarter.
 6. Tesla continues to lead in electric vehicle market share globally.
 7. Tesla's Full Self-Driving revenue increased significantly.
 8. Tesla stock price reached new highs following earnings announcement.
 9. Tesla's energy storage business grew 40% year-over-year.
10. Tesla Model Y became the best-selling vehicle worldwide.

ANALYSIS:
✅ Hybrid Search: Mixed relevant and irrelevant results
✅ Reranking: Most relevant Tesla financial/production info at top
✅ Notice how reranking moved the most contextually relevant chunks higher

KEY IMPROVEMENTS AFTER RERANKING:
----------------------------------------
BEFORE (Hybrid Top 3):
  1. Tesla

In [16]:
# Combine the query and the relevant document contents
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

top_reranked = reranked_docs[:5]

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say so.
"""

# Create a local Chat model (no API key needed)
model = ChatHuggingFace(
    llm=HuggingFacePipeline.from_model_id(
        model_id="Qwen/Qwen2.5-1.5B-Instruct",
        task="text-generation",
        pipeline_kwargs={
            "max_new_tokens": 512,
            "do_sample": False,
            "return_full_text": False,
        },
    )
)

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

# Display the full result and content only
print("\n--- Generated Response ---")
# print("Full result:")
# print(result)
print("Content only:")
print(result.content)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 9222.39it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces be


--- Generated Response ---
Content only:
Tesla reported record quarterly revenue of $25.2 billion in Q3 2024, indicating strong financial performance. Additionally, Tesla generated strong free cash flow of $7.5 billion, further supporting its financial health. The company also reported an improvement in its automotive gross margin to 19.3%, suggesting positive developments in its manufacturing processes and cost management strategies.
